# Multi-Agent SAC (MARL) — Independent Learners cho Smart Grid

**Giai đoạn 1** trong lộ trình phát triển: `Centralized SAC → MARL (IL) → Federated RL`

## Kiến trúc tổng quan

```
Centralized (baseline):              MARL — Independent Learners:
        Agent                    Agent₁   Agent₂  ...  Agent₆
          ↓                        ↓        ↓             ↓
  obs ∈ ℝ²⁴⁶                  obs₁∈ℝ⁴¹  obs₂∈ℝ⁴¹   obs₆∈ℝ⁴¹
  act ∈ ℝ¹⁸                   act₁∈ℝ³   act₂∈ℝ³    act₆∈ℝ³
          ↓                        ↓        ↓             ↓
  [B1,B2,B3,B4,B5,B6]            [B1]     [B2]         [B6]
```

## Lý thuyết toán học

### 1. Bài toán Dec-POMDP
MARL được mô hình hóa như **Decentralized Partially Observable MDP (Dec-POMDP)**:
$$\\langle \\mathcal{I}, \\mathcal{S}, \\{\\mathcal{A}^i\\}, \\mathcal{P}, \\{\\mathcal{R}^i\\}, \\{\\Omega^i\\}, \\mathcal{O}, \\gamma \\rangle$$

| Ký hiệu | Ý nghĩa |
|---|---|
| $\\mathcal{I} = \\{1,...,N\\}$ | Tập N agent (N = 6 tòa nhà) |
| $s \\in \\mathcal{S}$ | Trạng thái toàn cục (ẩn) |
| $o^i \\in \\Omega^i$ | Quan sát cục bộ của agent $i$ (41 chiều) |
| $a^i \\in \\mathcal{A}^i$ | Hành động của agent $i$ (3 chiều) |
| $r^i$ | Phần thưởng agent $i$ nhận được |

### 2. Independent Learners (IL)
Mỗi agent $i$ tự học policy $\\pi^i$ **độc lập**, coi các agent khác là một phần của môi trường:

$$\\pi^i_* = \\arg\\max_{\\pi^i} \\mathbb{E}_{\\tau \\sim \\pi^i} \\left[ \\sum_{t=0}^T r^i_t + \\alpha \\mathcal{H}(\\pi^i(\\cdot|o^i_t)) \\right]$$

**Ưu điểm:**
- Observation space nhỏ hơn: $41$ vs $246$ chiều → hội tụ nhanh hơn
- Scalable: thêm nhà mới không cần train lại toàn bộ
- Nền tảng tự nhiên cho **Federated RL** (thêm aggregation là xong)

**Hạn chế (Non-stationarity problem):**
Khi agent $i$ đang học, môi trường **thay đổi** vì các agent $j \\neq i$ cũng cập nhật policy → vi phạm giả định Markov. Đây là vấn đề cốt lõi mà FedRL giải quyết qua periodic aggregation.

### 3. Quan hệ với Centralized SAC
```
Centralized SAC = MARL với shared policy và joint obs/action
IL-MARL        = N SAC độc lập, mỗi cái có policy riêng
FedRL          = IL-MARL + periodic weight aggregation
```

In [ ]:
import os
import glob
import copy
from pathlib import Path
import numpy as np
import pandas as pd

# SB3 Imports
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.monitor import Monitor

# CityLearn Imports
from citylearn.citylearn import CityLearnEnv
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

In [ ]:
# ==============================================================
# GLOBAL PARAMETERS
# ==============================================================
NUM_HOUSES       = 6       # Số tòa nhà tham gia MARL
NUM_CPU          = 2       # Số CPU workers cho SubprocVecEnv mỗi agent
EPISODE_LENGTH   = 168     # 1 tuần (giờ)

# SAC Hyperparameters — giữ nhất quán với single_house_sac baseline
NET_ARCH         = [256, 256]
BATCH_SIZE       = 256
BUFFER_SIZE      = 100_000
LEARNING_STARTS  = 1_000   # Bước random explore trước khi train
TOTAL_TIMESTEPS  = 600_000 # Timesteps huấn luyện mỗi agent
SAVE_EVERY_EP    = 200     # Lưu checkpoint sau mỗi N episodes

BASE_SCHEMA      = "citylearn_challenge_2023_phase_3_1"
SESSION_NAME     = "marl_sac_independent_learners"

TENSORBOARD_DIR  = Path.cwd() / "training_logs"
MODEL_DIR        = Path.cwd() / "models" / SESSION_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f"  MARL — Independent Learners")
print(f"  {NUM_HOUSES} agents | {NUM_CPU} workers/agent | {TOTAL_TIMESTEPS:,} steps/agent")
print(f"  Total training steps: {NUM_HOUSES * TOTAL_TIMESTEPS:,}")
print(f"{'='*60}")

## Xây dựng môi trường per-building

Mỗi agent cần một **môi trường riêng** chỉ chứa tòa nhà của mình:

```
full_schema (6 buildings)
    ↓ filter
schema_B1  schema_B2  ...  schema_B6
    ↓          ↓                ↓
 Env_B1     Env_B2          Env_B6
    ↓          ↓                ↓
 SAC_B1     SAC_B2          SAC_B6
```

**NormalizedObservationWrapper** áp dụng min-max per feature:
$$\\hat{o}^i_k = \\frac{o^i_k - o^{i,min}_k}{o^{i,max}_k - o^{i,min}_k} \\in [0,1]$$

**SubprocVecEnv** chạy `NUM_CPU` bản copy song song để thu thập dữ liệu nhanh hơn, giảm variance gradient.

> **Lưu ý**: Mỗi agent train **độc lập** — không biết gì về hành động/trạng thái của các agent khác trong quá trình training. Đây là "Independent" trong Independent Learners.

In [ ]:
def make_env(schema_dict, episode_time_steps):
    """
    Factory function tạo môi trường CityLearn cho 1 tòa nhà.
    Dùng cho SubprocVecEnv — mỗi worker là 1 bản copy độc lập.
    """
    def _init():
        env = CityLearnEnv(
            schema=schema_dict,
            central_agent=True,
            episode_time_steps=episode_time_steps
        )
        env = NormalizedObservationWrapper(env)   # obs → [0,1]
        env = StableBaselines3Wrapper(env)         # gym-compatible API
        env = Monitor(env)                         # ghi lại ep_reward, ep_len
        return env
    return _init


def get_single_building_schema(full_schema, building_name):
    """Lọc schema chỉ giữ lại 1 tòa nhà duy nhất."""
    schema = copy.deepcopy(full_schema)
    schema["buildings"] = {building_name: schema["buildings"][building_name]}
    return schema


# ------------------------------------------------------------------
# Tải schema gốc và lấy danh sách tòa nhà
# ------------------------------------------------------------------
print("Đang tải schema...")
_temp_env = CityLearnEnv(schema=BASE_SCHEMA)
FULL_SCHEMA = copy.deepcopy(_temp_env.schema)
ALL_BUILDING_NAMES = list(FULL_SCHEMA["buildings"].keys())
BUILDING_NAMES = ALL_BUILDING_NAMES[:NUM_HOUSES]
del _temp_env

print(f"Tất cả các tòa nhà trong dataset : {ALL_BUILDING_NAMES}")
print(f"Tòa nhà được chọn ({NUM_HOUSES} nhà): {BUILDING_NAMES}")

# Tạo per-building schema dict
BUILDING_SCHEMAS = {
    name: get_single_building_schema(FULL_SCHEMA, name)
    for name in BUILDING_NAMES
}

# ------------------------------------------------------------------
# Kiểm tra kích thước obs/action của 1 tòa nhà
# ------------------------------------------------------------------
_probe_env = make_env(BUILDING_SCHEMAS[BUILDING_NAMES[0]], EPISODE_LENGTH)()
OBS_DIM_PER_AGENT = _probe_env.observation_space.shape[0]
ACT_DIM_PER_AGENT = _probe_env.action_space.shape[0]
_probe_env.close()

print(f"\nPer-agent observation dim : {OBS_DIM_PER_AGENT}")
print(f"Per-agent action dim      : {ACT_DIM_PER_AGENT}")
print(f"Joint obs dim (6 agents)  : {OBS_DIM_PER_AGENT * NUM_HOUSES}")
print(f"Joint act dim (6 agents)  : {ACT_DIM_PER_AGENT * NUM_HOUSES}")

## Khởi tạo các SAC Agent

Mỗi agent là một **SAC độc lập** với:
- Policy network: `[256, 256]` (giống `single_house_sac` baseline để so sánh công bằng)
- Replay buffer riêng: `100,000` transitions
- Entropy coefficient $\\alpha$ tự điều chỉnh

**Checkpoint resume**: Nếu đã có checkpoint từ lần train trước, tự động tiếp tục từ đó (kể cả replay buffer).

### Cấu trúc thư mục lưu model:
```
models/marl_sac_independent_learners/
├── Building_1/
│   ├── sac_Building_1_XXXXX_steps.zip
│   ├── sac_Building_1_XXXXX_steps_replay_buffer.pkl
│   └── sac_Building_1_final.zip
├── Building_2/
│   └── ...
└── Building_6/
    └── ...
```

In [ ]:
agents    = {}   # {building_name: SAC model}
vec_envs  = {}   # {building_name: SubprocVecEnv}

for building_name in BUILDING_NAMES:
    print(f"\n[{building_name}] Đang khởi tạo...")

    # ------ Tạo vectorized environment ------
    schema_i = BUILDING_SCHEMAS[building_name]
    env_fns  = [make_env(schema_i, EPISODE_LENGTH) for _ in range(NUM_CPU)]
    vec_env  = SubprocVecEnv(env_fns)
    vec_envs[building_name] = vec_env

    # ------ Thư mục lưu agent ------
    agent_dir = MODEL_DIR / building_name
    agent_dir.mkdir(parents=True, exist_ok=True)

    # ------ Kiểm tra checkpoint ------
    existing_ckpts = glob.glob(str(agent_dir / "sac_*.zip"))
    # Loại bỏ replay buffer files (không phải model)
    existing_ckpts = [f for f in existing_ckpts if "replay_buffer" not in f]

    if existing_ckpts:
        latest_ckpt = max(existing_ckpts, key=os.path.getctime)
        print(f"  → Resume từ checkpoint: {Path(latest_ckpt).name}")
        agent = SAC.load(
            latest_ckpt,
            env=vec_env,
            tensorboard_log=str(TENSORBOARD_DIR)
        )
        # Tải lại replay buffer nếu có
        rb_path = latest_ckpt.replace(".zip", "_replay_buffer.pkl")
        if os.path.exists(rb_path):
            agent.load_replay_buffer(rb_path)
            print(f"  → Replay buffer loaded ({agent.replay_buffer.size():,} transitions)")
        reset_steps = False

    else:
        print(f"  → Khởi tạo model mới")
        agent = SAC(
            "MlpPolicy",
            vec_env,
            policy_kwargs=dict(net_arch=NET_ARCH),
            batch_size=BATCH_SIZE,
            buffer_size=BUFFER_SIZE,
            learning_starts=LEARNING_STARTS,
            verbose=1,
            tensorboard_log=str(TENSORBOARD_DIR),
        )
        reset_steps = True

    # Gắn flag reset_num_timesteps để dùng lúc .learn()
    agent._reset_steps_flag = reset_steps
    agents[building_name] = agent

print(f"\n{'='*50}")
print(f"Đã khởi tạo {len(agents)} agents thành công.")

# In tóm tắt kiến trúc mạng của agent đầu tiên
first_agent = agents[BUILDING_NAMES[0]]
print(f"\nPolicy network (actor):")
print(first_agent.policy.actor)
print(f"\nCritic network (Q1):")
print(first_agent.policy.critic)

## Training — Huấn luyện tuần tự từng agent

Mỗi agent huấn luyện **độc lập** trên môi trường 1-nhà của mình.

**Tại sao train tuần tự (không song song)?**  
Trong notebook này, các agent train lần lượt để đơn giản hóa. Trong thực tế production (và trong FedRL), mỗi agent chạy trên **node độc lập** → hoàn toàn song song.

**Checkpoint strategy:**
$$\\text{save\\_freq\\_steps} = \\left\\lfloor \\frac{T_{ep} \\times N_{save\\_ep}}{N_{cpu}} \\right\\rfloor$$
$$= \\left\\lfloor \\frac{168 \\times 200}{2} \\right\\rfloor = 16{,}800 \\text{ steps}$$

**TensorBoard**: Mỗi agent log riêng với tên `{session}_{building_name}` để so sánh:
```bash
tensorboard --logdir training_logs/
```

In [ ]:
save_freq_steps = max((EPISODE_LENGTH * SAVE_EVERY_EP) // NUM_CPU, 1)
print(f"Checkpoint mỗi {save_freq_steps:,} steps (~{SAVE_EVERY_EP} episodes)\n")

for idx, building_name in enumerate(BUILDING_NAMES):
    print(f"\n{'='*60}")
    print(f"  [{idx+1}/{NUM_HOUSES}] Training agent: {building_name}")
    print(f"{'='*60}")

    agent     = agents[building_name]
    agent_dir = MODEL_DIR / building_name

    checkpoint_cb = CheckpointCallback(
        save_freq=save_freq_steps,
        save_path=str(agent_dir),
        name_prefix=f"sac_{building_name}",
        save_replay_buffer=True,   # Lưu replay buffer để resume chính xác
        save_vecnormalize=False,
    )

    agent.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=checkpoint_cb,
        tb_log_name=f"{SESSION_NAME}_{building_name}",
        reset_num_timesteps=agent._reset_steps_flag,
        log_interval=1,   # Log sau mỗi episode
    )

    # ------ Lưu final model ------
    final_path = agent_dir / f"sac_{building_name}_final"
    agent.save(str(final_path))
    agent.save_replay_buffer(str(agent_dir / f"sac_{building_name}_final_replay_buffer"))
    print(f"  ✓ Saved: {final_path}.zip")

print(f"\n{'='*60}")
print("  TRAINING COMPLETE — Tất cả {NUM_HOUSES} agents đã được huấn luyện.")
print(f"{'='*60}")

## Evaluate — Đánh giá MARL trên môi trường joint (6 nhà)

Đây là bước **quan trọng nhất** của MARL: các agent **chưa bao giờ gặp nhau** trong training, nhưng giờ phải phối hợp trên môi trường thực 6-nhà.

### Cơ chế split-obs / combine-actions

```
Joint obs (246-dim)
    │
    ├─ obs[0:41]   → Agent_1 → action_1 (3-dim) ─┐
    ├─ obs[41:82]  → Agent_2 → action_2 (3-dim)  │
    ├─ obs[82:123] → Agent_3 → action_3 (3-dim)  ├─ joint_action (18-dim) → env.step()
    ├─ obs[123:164]→ Agent_4 → action_4 (3-dim)  │
    ├─ obs[164:205]→ Agent_5 → action_5 (3-dim)  │
    └─ obs[205:246]→ Agent_6 → action_6 (3-dim) ─┘
```

Công thức tổng quát:
$$o^i = o_{joint}[i \\cdot d_{obs} : (i+1) \\cdot d_{obs}], \\quad d_{obs} = 41$$
$$a_{joint} = \\text{concat}(a^1, a^2, ..., a^N), \\quad a^i = \\pi^i(o^i)$$

> Khi evaluate dùng `deterministic=True` — tắt noise Gaussian, chỉ dùng mean $\\mu_\\theta(o^i)$.

In [ ]:
# ==============================================================
# 1. TẠO JOINT EVALUATION ENVIRONMENT (6 nhà)
# ==============================================================
joint_schema = copy.deepcopy(FULL_SCHEMA)
joint_schema["buildings"] = {
    name: joint_schema["buildings"][name] for name in BUILDING_NAMES
}

# Không dùng episode_time_steps → chạy toàn bộ dataset
eval_env_raw  = CityLearnEnv(joint_schema, central_agent=True)
eval_env_norm = NormalizedObservationWrapper(eval_env_raw)
eval_env      = StableBaselines3Wrapper(eval_env_norm)

# Kiểm tra kích thước
joint_obs_dim = eval_env.observation_space.shape[0]
joint_act_dim = eval_env.action_space.shape[0]
obs_per_agent = joint_obs_dim // NUM_HOUSES
act_per_agent = joint_act_dim // NUM_HOUSES

print(f"Joint observation space : {eval_env.observation_space}")
print(f"Joint action space      : {eval_env.action_space}")
print(f"Obs per agent           : {obs_per_agent}")
print(f"Act per agent           : {act_per_agent}")

# ==============================================================
# 2. TẢI FINAL AGENTS
# ==============================================================
print("\nĐang tải final agents...")
loaded_agents = {}
for building_name in BUILDING_NAMES:
    model_path = MODEL_DIR / building_name / f"sac_{building_name}_final.zip"
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model không tồn tại: {model_path}\n"
            "Hãy chạy cell Training trước."
        )
    loaded_agents[building_name] = SAC.load(str(model_path))
    print(f"  ✓ Loaded: {building_name}")

In [ ]:
# ==============================================================
# 3. EVALUATION ROLLOUT
# ==============================================================
obs, _info = eval_env.reset()
done        = False
total_reward = 0.0
step_count   = 0

print("Đang chạy deterministic rollout trên joint environment...")
print(f"{'─'*50}")

while not done:
    # ── Bước 1: Tách obs theo từng agent ──────────────────────
    obs_per_building = [
        obs[i * obs_per_agent : (i + 1) * obs_per_agent]
        for i in range(NUM_HOUSES)
    ]

    # ── Bước 2: Mỗi agent dự đoán action của mình ─────────────
    action_parts = []
    for i, building_name in enumerate(BUILDING_NAMES):
        agent_i   = loaded_agents[building_name]
        obs_i     = obs_per_building[i].reshape(1, -1)
        # deterministic=True → dùng mean policy μ(o), không sample
        action_i, _ = agent_i.predict(obs_i, deterministic=True)
        action_parts.append(action_i[0])

    # ── Bước 3: Ghép lại thành joint action ───────────────────
    joint_action = np.concatenate(action_parts, axis=0).astype(np.float32)

    # ── Bước 4: Bước môi trường ───────────────────────────────
    step_result = eval_env.step(joint_action)
    if len(step_result) == 5:
        obs, reward, terminated, truncated, info = step_result
        done = terminated or truncated
    else:
        obs, reward, done, info = step_result

    r = reward[0] if hasattr(reward, "__len__") else float(reward)
    total_reward += r
    step_count   += 1

print(f"Rollout hoàn tất!")
print(f"  Tổng bước     : {step_count:,}")
print(f"  Total reward  : {total_reward:.4f}")
print(f"  Avg reward/step: {total_reward / step_count:.4f}")

## KPI Extraction & So sánh với Baseline

### Các KPI chính cần quan tâm

| KPI | Ý nghĩa | Tốt khi |
|---|---|---|
| `cost_total` | Chi phí điện (normalized) | < 1.0 |
| `discomfort_proportion` | % thời gian nhiệt độ ngoài setpoint | Thấp |
| `carbon_emissions_total` | Phát thải CO₂ (normalized) | < 1.0 |
| `electricity_consumption_total` | Điện tiêu thụ (normalized) | < 1.0 |
| `zero_net_energy` | Tiến tới ZNE (0 = perfect) | < 1.0 |

**Normalization**: Tất cả KPI được normalize so với **baseline không có agent** → giá trị < 1.0 nghĩa là tốt hơn baseline, > 1.0 là tệ hơn.

In [ ]:
# ==============================================================
# 4. KPI EXTRACTION
# ==============================================================
kpis = eval_env_raw.evaluate()
kpis_pivot = (
    kpis
    .pivot(index="cost_function", columns="name", values="value")
    .round(4)
    .dropna(how="all")
)

print("=== MARL — Independent Learners KPIs ===")
display(kpis_pivot)

In [ ]:
# ==============================================================
# 5. BẢNG SO SÁNH TOÀN DIỆN
# ==============================================================

# Kết quả từ baseline notebooks (đã có trong repo)
BASELINE_RESULTS = {
    "RBC (1 nhà)": {
        "cost_total":                       2.013,
        "discomfort_proportion":            0.984,
        "carbon_emissions_total":           2.141,
        "electricity_consumption_total":    2.121,
        "zero_net_energy":                  2.199,
        "all_time_peak_average":            1.138,
        "ramping_average":                  1.091,
    },
    "SAC Centralized (1 nhà)": {
        "cost_total":                       0.8959,
        "discomfort_proportion":            0.0657,
        "carbon_emissions_total":           0.9293,
        "electricity_consumption_total":    0.9282,
        "zero_net_energy":                  0.8210,
        "all_time_peak_average":            1.0313,
        "ramping_average":                  1.9068,
    },
    "SAC Centralized (6 nhà)": {
        "cost_total":                       0.7947,
        "discomfort_proportion":            0.5415,
        "carbon_emissions_total":           0.8016,
        "electricity_consumption_total":    0.7991,
        "zero_net_energy":                  0.7427,
        "all_time_peak_average":            0.9460,
        "ramping_average":                  1.1309,
    },
}

# Lấy KPI của MARL từ kết quả evaluate (cột District)
KEY_METRICS = [
    "cost_total",
    "discomfort_proportion",
    "carbon_emissions_total",
    "electricity_consumption_total",
    "zero_net_energy",
    "all_time_peak_average",
    "ramping_average",
]

marl_district = {}
for metric in KEY_METRICS:
    try:
        marl_district[metric] = kpis_pivot.loc[metric, "District"]
    except KeyError:
        marl_district[metric] = float("nan")

BASELINE_RESULTS["MARL IL (6 nhà) ← YOU ARE HERE"] = marl_district

# Xây dựng DataFrame so sánh
comparison_df = pd.DataFrame(BASELINE_RESULTS, index=KEY_METRICS)
comparison_df.index.name = "KPI"

# Highlight: giá trị thấp nhất trong mỗi row (tốt nhất)
def highlight_best(s):
    is_best = s == s.min()
    return ["background-color: #c8f7c5; font-weight: bold" if v else "" for v in is_best]

styled = (
    comparison_df
    .style
    .apply(highlight_best, axis=1)
    .format("{:.4f}", na_rep="N/A")
    .set_caption("So sánh các phương pháp điều khiển (giá trị thấp = tốt hơn)")
)

print("\n=== BẢNG SO SÁNH ===")
display(styled)

In [ ]:
# ==============================================================
# 6. PHÂN TÍCH PER-BUILDING
# ==============================================================
print("=== KPI từng tòa nhà (MARL IL) ===\n")

# Chỉ lấy các cột building (không lấy District)
per_building_cols = [c for c in kpis_pivot.columns if c != "District"]
per_building_kpis = kpis_pivot[per_building_cols].loc[
    [m for m in KEY_METRICS if m in kpis_pivot.index]
]

display(per_building_kpis)

# Tính variance giữa các agent → đo độ "fair" của MARL
print("\n=== Variance giữa các tòa nhà (cao = bất bình đẳng) ===")
variance_df = per_building_kpis.var(axis=1).round(6).rename("variance")
print(variance_df.to_string())

## Kết luận & Bước tiếp theo

### MARL vs Centralized SAC — Phân tích

| Góc độ | MARL Independent Learners | SAC Centralized |
|---|---|---|
| **Observation space** | 41-dim/agent (nhỏ, hội tụ nhanh) | 246-dim (lớn, khó train) |
| **Privacy** | Mỗi nhà giữ data riêng | Phải share toàn bộ obs |
| **Scalability** | Thêm nhà = thêm agent mới | Phải train lại từ đầu |
| **Coordination** | Không có (Independent) | Có (joint policy) |
| **Phù hợp với FedRL** | ✓ Tự nhiên | ✗ Không |

### Non-stationarity Problem

MARL IL có điểm yếu cốt lõi: khi agent $i$ đang học, môi trường của nó **không dừng (non-stationary)** vì agent $j$ cũng đang thay đổi:

$$P(s_{t+1} | s_t, a^i_t) \\neq P(s_{t+1} | s_t, a^i_t, a^{-i}_t)$$

Agent $i$ không quan sát được $a^{-i}_t$ → vi phạm Markov assumption của SAC.

### Bước tiếp theo → Federated RL

```
MARL IL (hiện tại)
    ↓  + periodic weight aggregation
FedRL
    ↓
w_global = Σ (n_i/n) × w_i   [FedAvg]
```

Giải pháp non-stationarity trong FedRL:
- **Synchronous aggregation**: Tất cả agent đồng bộ hóa sau mỗi $K$ episodes
- **Shared global policy**: $w_{global}$ mang thông tin của tất cả buildings
- **Local fine-tuning**: Mỗi agent fine-tune từ $w_{global}$ → personalization